# **Lab 3. Prompt Engineering with Command-A.**

**About OCI Generative AI  Service**

OCI Generative AI is a fully managed service that provides a set of state-of-the-art, customizable large language models (LLMs) that cover a wide range of use cases, and which is available through a single API.
Using the OCI Generative AI service you can access ready-to-use pretrained models, or create and host your own fine-tuned custom models based on your own data on dedicated AI clusters. Detailed documentation of the service and API is available __[here](https://docs.oracle.com/en-us/iaas/Content/generative-ai/home.htm)__ and __[here](https://docs.oracle.com/en-us/iaas/api/#/en/generative-ai/20231130/)__.

In this lab, we shall learn to use prompting principles.

# Sequence 1. Setup Access to Cohere Models

## OCI Generative AI Service
We will use OCI GenAI's `cohere.command-a` model. 

Import the necessary libraries

In [1]:
! cp oci ~/.oci

cp: -r not specified; omitting directory 'oci'


In [2]:
import oci
from LoadProperties import LoadProperties
# Setup basic variables
properties = LoadProperties()

In [3]:
import oci

CONFIG_PROFILE = "DEFAULT"
config = oci.config.from_file('~/.oci/config', CONFIG_PROFILE)

# Service endpoint
endpoint = properties.getEndpoint()

# Generative AI Service Client
GenAI_client = oci.generative_ai_inference.GenerativeAiInferenceClient(config=config, service_endpoint=endpoint, retry_strategy=oci.retry.NoneRetryStrategy(), timeout=(10,240))

### OCI Generative AI Service (From Instance on OCI - Reference only)

#### Use Instance Principals for Authentication and Initialize the Generative AI Client
- Using instance principal authentication, you can authorize an instance to make API calls on Oracle Cloud Infrastructure services. 
- It removes the need to configure user credentials or a configuration file.

- The client will be used to make requests to the OCI Generative AI service.
- `config={}` is used to specify the configuration for connecting to OCI. 
- An empty dictionary {} is provided because authentication is handled using signer in the previous step, making it unnecessary to provide a detailed configuration file.
- `properties.getEndpoint()` retrieves this URL from a configuration file (`config.txt`), using a method defined in the LoadProperties class. The model is hosted at this URL.
- `oci.retry.NoneRetryStrategy()` means that if a request fails, the client will not automatically retry it.
- `timeout=(10, 240)` specifies the timeouts for the client. 10 seconds is the connection timeout, meaning how long the client waits for a connection to be established with the server. 240 seconds is the read timeout, meaning how long the client waits for a response after a connection is established.

## Create a helper function to use prompts and return the generated outputs.  

### Understanding the `get_completion` Function

The `get_completion` function is designed to interact with Oracle Cloud Infrastructure (OCI) Generative AI Inference services. It takes a `prompt` as input and returns a generated text based on that prompt.

#### Step-by-Step Breakdown

1.  **Initialization of Models**: The function initializes several models from the  `oci.generative_ai_inference.models`  module:
    
    -   `cr.message`: Content of the message.
    -   `CohereChatRequest`: To configure the request for Cohere Chat, including API format, messages, and parameters controlling generation.
2.  **Setting Up Request Parameters**:
    
    -   The request is configured with specific parameters:
        -   `max_tokens`: Set to 1600, which controls the maximum number of tokens in the response.
        -   `temperature` and  `frequency_penalty`: These control aspects of how creative or repetitive the generated text can be. Specifically:
            -   Temperature: Set to 0 for deterministic output (no randomness).
            -   Frequency penalty: Set to 1, which means that repeating tokens will be penalized more heavily.
        -   `top_p`: Set to 0.75, controlling diversity by specifying that only tokens with cumulative probability up to this value should be considered.
3.  **Serving Mode and Compartment ID**:
    
    -   The serving mode is set to on-demand with a model ID retrieved from properties (`properties.getModelName()`).
    -   The compartment ID is also retrieved from properties (`properties.getCompartment()`).
4.  **Sending Request and Retrieving Response**:
    
    -   A client object (`generative_ai_inference_client`) sends a chat request based on configured details (`cd`).
    -   The response from this request contains generated text based on the input prompt.
5.  **Returning Generated Text**:
    
    -   The function extracts and returns the first choice's message content's text from the response data.
```

In [4]:
def get_completion(prompt):
  cd = oci.generative_ai_inference.models.ChatDetails()

  cr = oci.generative_ai_inference.models.CohereChatRequest()
  cr.message = prompt
  cr.max_tokens = 600
  cr.temperature = 0
  cr.frequency_penalty = 1
  cr.top_p = 0.75
  cr.top_k = 0

  cd.serving_mode = oci.generative_ai_inference.models.OnDemandServingMode(model_id=properties.getModelName())
  cd.chat_request = cr
  cd.compartment_id = properties.getCompartment()
  cr = GenAI_client.chat(cd)

  return (cr.data.chat_response.text)

# Sequence 2. Prompting Principles
## Principle 1. Write clear and specific instructions
### 1. Use delimiters to clearly indicate distinct parts of the input
- Delimiters can be anything like: ```, """, < >, `<tag> </tag>`, `:`

In [5]:
text = f"""
You should express what you want a model to do by providing instructions that are as clear and \
specific as you can possibly make them. This will guide the model towards the desired output, \
and reduce the chances of receiving irrelevant or incorrect responses. Don't confuse writing a \
clear prompt with writing a short prompt. In many cases, longer prompts provide more clarity \
and context for the model, which can lead to more detailed and relevant outputs.
"""
prompt = f"""
Summarize the text delimited by triple backticks into a single sentence.
```{text}```
"""
response = get_completion(prompt)
print(response)

To ensure accurate results, write detailed and specific instructions for a model, balancing clarity with context to guide it towards the desired output.


### 2: Ask for a structured output
- JSON, HTML

In [ ]:
prompt = f"""Generate a list of three made-up book titles along with their authors and genres. 
Provide them in JSON format with the following keys: 
book_id, title, author, genre.
"""
response = get_completion(prompt)
print(response)

### 3. Ask the model to check whether conditions are satisfied

In [ ]:
text_1 = f"""
Making a cup of tea is easy! First, you need to get some water boiling. While that's happening, grab a cup and put a tea bag in it. 
Once the water is hot enough, just pour it over the tea bag. Let it sit for a bit so the tea can steep. After a few minutes, take out 
the tea bag. If you like, you can add some sugar or milk to taste. And that's it! You've got yourself a delicious cup of tea to enjoy.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

In [ ]:
text_2 = f"""
The sun is shining brightly today, and the birds are singing. It's a beautiful day to go for a walk in the park. 
The flowers are blooming, and the trees are swaying gently in the breeze. People are out and about, enjoying the lovely weather. 
Some are having picnics, while others are playing games or simply relaxing on the grass. It's a perfect day to spend time 
outdoors and appreciate the beauty of nature.
"""
prompt = f"""
You will be provided with text delimited by triple quotes. 
If it contains a sequence of instructions, re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, then simply write \"No steps provided.\"

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

### 4. "Few-shot" prompting

In [ ]:
prompt = f"""
Your task is to answer in a consistent style. Respond to one question only.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest valley flows from a modest spring; the grandest symphony originates from a single note; 
the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
"""
response = get_completion(prompt)
print(response)

## Principle 2: Give the model time to “think”

### 1. Specify the steps required to complete a task

In [ ]:
text = f"""In a charming village, siblings Jack and Jill set out on a quest to fetch water from a hilltop well. As they climbed, 
singing joyfully, misfortune struck—Jack tripped on a stone and tumbled down the hill, with Jill following suit. 
Though slightly battered, the pair returned home to comforting embraces. Despite the mishap, their adventurous spirits remained undimmed, 
and they continued exploring with delight.
"""
# example 1
prompt_1 = f"""
Perform the following actions: 
1 - Summarize the following text delimited by triple backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following keys: french_summary, num_names.

Separate your answers with line breaks.

Text:
```{text}```
"""
response = get_completion(prompt_1)
print("Completion for prompt 1:")
print(response)

#### 2. Ask for output in a specified format

In [ ]:
prompt_2 = f"""Your task is to perform the following actions: 
1 - Summarize the following text delimited by <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with summary and num_names>

Text: <{text}>
"""
response = get_completion(prompt_2)
print("\nCompletion for prompt 2:")
print(response)

## Try experimenting on your own!

#### A note about the backslash
- In the lab, we are using a backslash `\` to make the text fit on the screen without inserting newline '\n' characters.
- The Model isn't really affected whether you insert newline characters or not.  But when working with LLMs in general, you may consider whether newline characters in your prompt may affect the model's performance.
### Try Python code examples present in the examples directory and create your own prompts. Extend the code to include user inputs and display outputs.
#### Congratulations! you have Successfully completed this Lab